# Setup

In [ ]:
%pip install faster-whisper ffmpeg-python --quiet

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))
recording_path = r"c:/temp/Recording.m4a"
second_recording_path = r"c:/temp/Recording 2.m4a"
log_entry_Date = "2024-06-02"
logs_path = os.path.abspath('../data/logs')

In [ ]:
import services.llm_client as llm_client
import services.transcriber as transcriber
import services.printer as printer
import services.database as database

# Main Execution

In [ ]:
database.reset_db() #Reset the database to ensure a clean slate for testing. 

import pathlib
if pathlib.Path(f'{logs_path}/{log_entry_Date}.md').is_file():
    pathlib.Path(f'{logs_path}/{log_entry_Date}.md').unlink() #Delete existing output.md if it exists to avoid confusion with previous runs. No idea why the is unlink and not something like remove().


In [ ]:
def print_oputput(transcription: str, markdown_response: str, questions_response: str):
    printer.print_section("Transcription", transcription)
    printer.print_section("Markdown formatted diary entry", markdown_response)
    printer.print_section("Follow up questions for self reflection", questions_response)
    print("=" * 88)

In [ ]:
def process_log_entry(entry_date, recording_path, print_output=False):
    #transcribe audio:
    transcription, audio_duration = transcriber.transcribe_audio(recording_path)

    #Create log entry header and segment:
    log_entry_id = database.create_or_get_log_header(entry_date)
    ls_transcript = database.create_log_segment(log_entry_id, str(recording_path), audio_duration, transcription)

    #LLM calls for markdown formatting and question generation:
    markdown_response = llm_client.llm_formatter(prompt=ls_transcript)

    #save markdown response as a .md file.
    with open(f"{logs_path}/{entry_date}.md", "w", encoding='utf-8') as f:
        f.write(markdown_response)

    questions_response = llm_client.llm_question_generator(prompt=markdown_response)

    database.create_log_enrichment(log_entry_id, markdown_response, questions_response)

    print("🎉 End to end process completed successfully!")
    if print_output:
        print_oputput(ls_transcript, markdown_response, questions_response)

In [ ]:
#process first log entry
process_log_entry("2024-06-02", recording_path, True)

In [ ]:
#process first log entry
process_log_entry("2024-06-02", second_recording_path, True)